In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🎓 Course Material Tutor

## What We're Building

A study assistant that indexes **lecture notes**, **slides**, and
**assignments** from a course and lets students ask questions. The tutor:

1. Retrieves relevant material from the course content
2. Explains concepts at the appropriate level
3. References which **lecture** and **week** the information comes from
4. Can quiz students with practice questions

## Why This Differs From Generic Q&A

A course tutor should:
- **Explain, not just answer** — students need to understand *why*
- **Point to the right lecture** — "revisit Lecture 5, slides 12-15"
- **Scaffold learning** — build on concepts from earlier weeks
- **Generate practice questions** — active recall beats passive reading

## Stack
- **LangChain** — orchestration + retrieval
- **ChromaDB** — vector store with lecture/week metadata
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Course Material

A simulated 4-week **Introduction to Machine Learning** course with
lecture notes, key concepts, and assignments.

In [ ]:
course_materials = [
    {
        "week": 1,
        "lecture": "Lecture 1: Introduction to ML",
        "material_type": "notes",
        "content": """LECTURE 1: INTRODUCTION TO MACHINE LEARNING

What is Machine Learning?
Machine learning is a subset of artificial intelligence where systems learn
patterns from data without being explicitly programmed. Instead of writing
rules by hand, we provide examples and let the algorithm discover the rules.

Types of Machine Learning:

1. Supervised Learning:
   - We have labeled data (inputs and correct outputs)
   - Goal: learn a function f(x) → y that maps inputs to outputs
   - Examples: classification (spam/not spam), regression (house prices)
   - Key algorithms: Linear Regression, Decision Trees, SVM, Neural Networks

2. Unsupervised Learning:
   - We have unlabeled data (inputs only, no correct outputs)
   - Goal: discover hidden structure in data
   - Examples: clustering (customer segments), dimensionality reduction
   - Key algorithms: K-Means, DBSCAN, PCA, Autoencoders

3. Reinforcement Learning:
   - Agent learns by interacting with an environment
   - Gets rewards/penalties for actions taken
   - Goal: maximize cumulative reward
   - Examples: game playing, robotics, autonomous driving

The ML Workflow:
1. Define the problem and success metrics
2. Collect and prepare data
3. Choose and train a model
4. Evaluate the model
5. Deploy and monitor

Key Concept: Generalization
The goal of ML is NOT to memorize training data. It's to generalize to
new, unseen data. A model that memorizes is "overfitting" — it performs
well on training data but poorly on test data.
""",
    },
    {
        "week": 1,
        "lecture": "Lecture 1: Introduction to ML",
        "material_type": "assignment",
        "content": """ASSIGNMENT 1: ML FUNDAMENTALS

Due: End of Week 1

Question 1 (10 pts): Classify each problem as supervised, unsupervised, or RL:
a) Predicting whether an email is spam based on past labeled emails
b) Grouping news articles by topic without predefined categories
c) Training a robot to walk by rewarding forward movement
d) Estimating house prices from features like area, bedrooms, location

Question 2 (15 pts): Explain the difference between overfitting and
underfitting. Draw a diagram showing training error and test error
as model complexity increases.

Question 3 (25 pts): Using the provided Iris dataset:
a) Split the data into 80% training and 20% test
b) Train a simple decision tree classifier
c) Report accuracy on both training and test sets
d) Does your model overfit? How can you tell?
""",
    },
    {
        "week": 2,
        "lecture": "Lecture 2: Linear Models",
        "material_type": "notes",
        "content": """LECTURE 2: LINEAR MODELS

Linear Regression:
The simplest ML model. Predicts a continuous output as a weighted sum of inputs.

Model: y = w0 + w1*x1 + w2*x2 + ... + wn*xn
- w0 is the bias (intercept)
- w1..wn are the feature weights
- The model learns weights that minimize the error

Loss Function — Mean Squared Error (MSE):
MSE = (1/n) * Σ(y_actual - y_predicted)²
This is the average squared difference between predictions and actual values.
We want to minimize this.

Gradient Descent:
Algorithm to find the weights that minimize the loss:
1. Start with random weights
2. Compute the gradient (direction of steepest increase)
3. Update weights in the opposite direction: w = w - learning_rate * gradient
4. Repeat until convergence

Learning Rate:
- Too high → overshoots the minimum, may diverge
- Too low → convergence is very slow
- Typical range: 0.001 to 0.1
- Advanced: learning rate schedulers that decrease over time

Regularization — Preventing Overfitting:
- L1 (Lasso): adds |w| penalty → some weights become exactly zero (feature selection)
- L2 (Ridge): adds w² penalty → weights are small but rarely zero
- ElasticNet: combines L1 and L2

Logistic Regression (for Classification):
Despite the name, it's a classifier. Outputs a probability using the sigmoid:
P(y=1|x) = 1 / (1 + e^(-z)) where z = w0 + w1*x1 + ... + wn*xn
- Output between 0 and 1 (probability)
- Decision boundary at P = 0.5 (configurable threshold)
- Uses cross-entropy loss instead of MSE
""",
    },
    {
        "week": 2,
        "lecture": "Lecture 2: Linear Models",
        "material_type": "assignment",
        "content": """ASSIGNMENT 2: LINEAR MODELS

Due: End of Week 2

Question 1 (20 pts): Implement linear regression from scratch:
a) Create a function that computes MSE loss
b) Implement gradient descent to learn weights
c) Plot the loss curve over iterations
d) Experiment with 3 different learning rates and describe the behavior

Question 2 (15 pts): Using scikit-learn's LinearRegression:
a) Fit a model on the Boston Housing dataset
b) Which features have the highest positive and negative weights?
c) What does the weight sign tell you about the feature's effect?

Question 3 (15 pts): Compare Ridge and Lasso on the same dataset:
a) Train both with alpha=1.0
b) How many features have zero weight in Lasso vs Ridge?
c) Which model has lower test MSE? Why?
""",
    },
    {
        "week": 3,
        "lecture": "Lecture 3: Decision Trees and Ensemble Methods",
        "material_type": "notes",
        "content": """LECTURE 3: DECISION TREES AND ENSEMBLE METHODS

Decision Trees:
A tree-structured model that makes decisions by splitting on features.

How Trees Split:
- At each node, find the feature and threshold that best separates the data
- Splitting criteria: Gini impurity or Information Gain (entropy)

Gini Impurity:
Gini = 1 - Σ(pi²) where pi is the proportion of class i
- Gini = 0 means pure node (all same class)
- Gini = 0.5 means maximum impurity (50/50 split for binary)

Information Gain (Entropy):
Entropy = -Σ(pi * log2(pi))
IG = Entropy(parent) - weighted_avg(Entropy(children))
- Choose the split that maximizes information gain

Tree Overfitting:
Trees can grow very deep and memorize training data.
Solutions:
- max_depth: limit tree height
- min_samples_leaf: minimum samples in a leaf node
- Pruning: grow full tree, then remove branches that don't help

Ensemble Methods — Combining Multiple Trees:

Random Forest:
- Train many trees on random subsets of data (bagging)
- Each tree also uses a random subset of features
- Final prediction = majority vote (classification) or average (regression)
- Reduces variance without increasing bias
- Hyperparameters: n_estimators, max_depth, max_features

Gradient Boosting (XGBoost, LightGBM, CatBoost):
- Train trees sequentially, each correcting the previous errors
- Each new tree fits the residual errors from the ensemble so far
- Learning rate controls how much each tree contributes
- Reduces bias — often achieves lowest error
- Hyperparameters: n_estimators, learning_rate, max_depth

Random Forest vs Gradient Boosting:
- RF: more robust, less tuning needed, parallelizable
- GB: often more accurate, but prone to overfitting, sequential
""",
    },
    {
        "week": 3,
        "lecture": "Lecture 3: Decision Trees and Ensemble Methods",
        "material_type": "assignment",
        "content": """ASSIGNMENT 3: TREES AND ENSEMBLES

Due: End of Week 3

Question 1 (15 pts): Decision Tree Visualization
a) Train a decision tree on the Iris dataset with max_depth=3
b) Visualize the tree using sklearn's plot_tree
c) Trace the decision path for a sample with sepal_length=5.0, sepal_width=3.5,
   petal_length=1.5, petal_width=0.2

Question 2 (20 pts): Random Forest vs Gradient Boosting
a) Using the wine quality dataset, compare RF and XGBoost
b) Try 3 values of n_estimators: 50, 100, 500
c) Plot test accuracy vs n_estimators for both models
d) Which model benefits more from additional trees? Why?

Question 3 (15 pts): Feature Importance
a) Extract feature importances from your best ensemble model
b) Plot a horizontal bar chart of importances
c) Are the top features the same for RF and XGBoost? Discuss.
""",
    },
    {
        "week": 4,
        "lecture": "Lecture 4: Model Evaluation and Selection",
        "material_type": "notes",
        "content": """LECTURE 4: MODEL EVALUATION AND SELECTION

Train/Test Split:
- Never evaluate on training data — always hold out a test set
- Common split: 80% train, 20% test
- For small datasets: use cross-validation instead

Cross-Validation:
- K-Fold CV: split data into k folds, train on k-1, test on 1, rotate
- Typical k=5 or k=10
- Gives more reliable estimate than a single train/test split
- Stratified K-Fold: preserves class proportions in each fold (use for classification)

Metrics for Classification:

Accuracy = (TP + TN) / Total
- Simple but misleading for imbalanced classes
- Example: 95% accuracy on 95/5 class split = just predicting majority class

Precision = TP / (TP + FP)
- Of all positive predictions, how many are correct?
- High precision = few false positives
- Important when false positives are costly (spam filter, fraud detection)

Recall = TP / (TP + FN)
- Of all actual positives, how many did we find?
- High recall = few false negatives
- Important when missing a positive is costly (disease detection)

F1 Score = 2 * (Precision * Recall) / (Precision + Recall)
- Harmonic mean of precision and recall
- Balances both metrics

ROC Curve and AUC:
- Plot True Positive Rate vs False Positive Rate at various thresholds
- AUC = Area Under Curve (1.0 = perfect, 0.5 = random)
- Threshold-independent evaluation

Metrics for Regression:
- MSE: Mean Squared Error (penalizes large errors)
- MAE: Mean Absolute Error (more robust to outliers)
- R²: Proportion of variance explained (1.0 = perfect)

Hyperparameter Tuning:
- Grid Search: try all combinations of parameters
- Random Search: sample random combinations (often faster)
- Bayesian Optimization: model the loss surface and pick smartly
- Always tune on validation set or CV, never on test set

The Bias-Variance Tradeoff:
- Bias: error from wrong assumptions (underfitting)
- Variance: error from sensitivity to training data (overfitting)
- As complexity increases: bias decreases, variance increases
- Optimal model: best tradeoff between bias and variance
""",
    },
    {
        "week": 4,
        "lecture": "Lecture 4: Model Evaluation and Selection",
        "material_type": "assignment",
        "content": """ASSIGNMENT 4: EVALUATION AND SELECTION (FINAL PROJECT)

Due: End of Week 4

OBJECTIVE: Build a complete ML pipeline on a real-world dataset.

Choose ONE of the following datasets:
a) Titanic survival prediction (binary classification)
b) California housing price prediction (regression)
c) Credit card fraud detection (imbalanced classification)

Requirements:

Part 1 — Data Exploration (15 pts)
- Summary statistics, missing values, class distribution
- At least 3 visualizations

Part 2 — Preprocessing (15 pts)
- Handle missing values (explain your strategy)
- Encode categorical features
- Scale numerical features if needed

Part 3 — Modeling (20 pts)
- Train at least 3 different models
- Use 5-fold cross-validation
- Report appropriate metrics (F1 for classification, R² for regression)

Part 4 — Analysis (10 pts)
- Compare models using a table of metrics
- Discuss which model you'd deploy and why
- What would you do differently with more time?

Grading:
- Code quality and documentation: 20%
- Methodology correctness: 40%
- Analysis depth: 30%
- Presentation: 10%
""",
    },
]

print(f"Loaded {len(course_materials)} course materials:")
for m in course_materials:
    print(f"  Week {m['week']} | {m['lecture'][:40]} | {m['material_type']}")

## Step 3 — Index Course Materials

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". "],
)

all_chunks = []
for material in course_materials:
    doc = Document(
        page_content=material["content"],
        metadata={
            "week": material["week"],
            "lecture": material["lecture"],
            "material_type": material["material_type"],
        },
    )
    chunks = splitter.split_documents([doc])
    all_chunks.extend(chunks)

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings,
    collection_name="course_tutor",
)

print(f"Indexed {vectorstore._collection.count()} chunks from {len(course_materials)} materials")

## Step 4 — Tutor: Explain Concepts

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.2)

tutor_prompt = ChatPromptTemplate.from_template(
    """You are a friendly and patient ML course tutor. Answer the student's
question using the course materials provided.

GUIDELINES:
- Explain concepts clearly, as if teaching a beginner
- Use analogies and examples to make abstract concepts concrete
- Reference the specific lecture and week: "As covered in Week X, Lecture Y..."
- If the question involves math, show the formula AND explain it intuitively
- Encourage the student — learning ML is hard and that's okay
- If relevant, suggest the related assignment for practice

Course Materials:
{context}

Student Question: {question}

Tutor Response:"""
)


def ask_tutor(question: str, week: int | None = None) -> None:
    """Ask the course tutor. Optionally limit to a specific week."""
    print(f"🙋 {question}")
    if week:
        print(f"   (searching Week {week} materials)")
    print("=" * 60)
    
    if week:
        results = vectorstore.similarity_search(
            question, k=5, filter={"week": week}
        )
    else:
        results = vectorstore.similarity_search(question, k=5)
    
    # Show sources
    print("\n📚 Materials referenced:")
    seen = set()
    for doc in results:
        key = (doc.metadata["lecture"], doc.metadata["material_type"])
        if key not in seen:
            seen.add(key)
            print(f"  Week {doc.metadata['week']} | {doc.metadata['lecture']} ({doc.metadata['material_type']})")
    print()
    
    context = "\n\n---\n\n".join(
        f"[Week {d.metadata['week']} | {d.metadata['lecture']} | {d.metadata['material_type']}]\n"
        f"{d.page_content}"
        for d in results
    )
    
    chain = tutor_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    print(answer)
    print("\n" + "=" * 60)


print("Course tutor ready!")

## Step 5 — Ask the Tutor

In [ ]:
ask_tutor("What's the difference between supervised and unsupervised learning?")

In [ ]:
ask_tutor("I don't understand gradient descent. Can you explain it simply?")

In [ ]:
ask_tutor("What's the difference between Random Forest and XGBoost?", week=3)

In [ ]:
ask_tutor("When should I use precision vs recall? I always get confused.")

In [ ]:
ask_tutor("What's the bias-variance tradeoff?")

## Step 6 — Quiz Generator

The tutor can also generate practice questions from the course material.

In [ ]:
quiz_prompt = ChatPromptTemplate.from_template(
    """You are a course tutor creating practice questions for students.
Generate 3 practice questions based on the course material below.

For each question:
- Mix difficulty levels: 1 easy, 1 medium, 1 hard
- Include the answer after each question (hidden behind a divider)
- Reference what concept is being tested
- Include one question that requires applying a concept, not just reciting

Course Material (Week {week}):
{context}

Topic: {topic}

Practice Questions:"""
)


def generate_quiz(topic: str, week: int) -> None:
    """Generate practice questions for a topic."""
    print(f"📝 Generating quiz for: {topic} (Week {week})")
    print("=" * 60)
    
    results = vectorstore.similarity_search(
        topic, k=4, filter={"week": week}
    )
    
    context = "\n\n".join(d.page_content for d in results)
    chain = quiz_prompt | llm | StrOutputParser()
    quiz = chain.invoke({"context": context, "topic": topic, "week": str(week)})
    
    print(quiz)
    print("\n" + "=" * 60)


print("Quiz generator ready!")

In [ ]:
generate_quiz("linear regression and gradient descent", week=2)

In [ ]:
generate_quiz("model evaluation metrics", week=4)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Week/lecture metadata** | Every chunk knows its week, lecture, and type (notes/assignment) |
| **Material type tagging** | Distinguishes between lecture notes and assignments |
| **Pedagogical prompting** | LLM is instructed to explain, use analogies, and encourage |
| **Week-scoped search** | Filter to specific weeks for targeted studying |
| **Quiz generation** | Create practice questions from the indexed material |

## 🔧 Customization Ideas

- **PDF slide loader**: Use `pypdf` to extract text from lecture slide PDFs
- **Difficulty adaptation**: Track student performance and adjust explanation level
- **Spaced repetition**: Surface material from earlier weeks for review
- **Socratic mode**: Instead of answering directly, ask guiding questions